In [2]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
v1 = model.encode(q1)

In [5]:
v1.shape

(384,)

In [6]:
v2 = model.encode(q2)

In [7]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)

In [8]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [9]:
v1.dot(dv)

np.float32(0.32332397)

In [10]:
q2 = 'How to install Docker on Windows?'
v2 = model.encode(q2)

In [11]:
v2.dot(dv)

np.float32(0.019730438)

In [12]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-25 09:44:27--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 

200 OK
Length: 888 [text/plain]
Saving to: ‘ingest.py.1’

ingest.py.1         100%[===================>]     888  --.-KB/s    in 0s      

2026-06-25 09:44:29 (34.7 MB/s) - ‘ingest.py.1’ saved [888/888]



In [13]:
from ingest import load_faq_data

documents = load_faq_data()

In [14]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [15]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [16]:
len(texts)

1350

In [17]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [18]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [19]:
import numpy as np
X = np.array(vectors)

In [20]:
scores = X.dot(v1)

In [21]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [22]:
documents[2]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [23]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]

In [24]:
scores[top5]

array([0.762941  , 0.7579371 , 0.7192132 , 0.6536312 , 0.56009996],
      dtype=float32)

In [25]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579371
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192132
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 

In [26]:
top5

array([  2, 625, 907, 538,   7])

In [27]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)